In [ ]:
from Microglia_Processing import *

In [ ]:
folder_path = '/Users/allisonpickle/Desktop/Image Processing Pipeline Analysis/All Microglia Copied/Background_Subtracted_Images'

In [ ]:
def Threshold_Test(folder_path):
    import os
    from skimage import io
    from skimage.filters import try_all_threshold
    import matplotlib.pyplot as plt
    """Test threshold options for images in a folder

    Parameters
    ----------
    folder_path : str
        Path to the folder containing the images

    Returns
    -------
    None
    """
    # Get a list of all image files in the folder
    image_files = [file for file in os.listdir(folder_path) if file.endswith('.tiff') or file.endswith('.tif')]

    # Iterate over each image file
    for image_file in image_files:
        # Construct the full path to the image file
        image_path = os.path.join(folder_path, image_file)

        # Read the image
        img = io.imread(image_path)
        fig, ax = try_all_threshold(img, figsize=(20, 18), verbose=False)
        plt.title(image_file)  # Add the image file name as the title
        plt.show()

# Example usage:
Threshold_Test('/Users/allisonpickle/Desktop/260204_Hyehyun_Aged_uglia_72hr_AB/200x/All/Microglia/Isolated microglia')

Before use: Isolate microglia channel into a new folder for processing

Autocrop microglia of interest for morphology assessment

In [ ]:
def Crop_Microglia(input_image_path, output_folder, padding=20, min_size_microns=100, bbox_list=None):
    """
    Takes an input image ending in _C_0.tiff, applies a mean filter, and crops the image
    around each individual thresholded object. Only crops objects that meet a minimum size
    determined by the user. If more than 5 objects exist, randomly selects 5 to process.
    Skips objects touching the edges of the image. Saves each cropped object as a separate
    image in the specified output folder with a _#_ added before the _C_0 in the filename.
    
    If bbox_list is provided, uses those bounding boxes instead of detecting new ones.
    Uses Li thresholding method.
    """
    import cv2
    import os
    import numpy as np
    import random
    from skimage.measure import regionprops, label as skimage_label
    from skimage.filters import threshold_li
    
    # Scale for converting microns² to pixels
    scale = 0.641  # microns per pixel
    min_size_pixels = int(min_size_microns / (scale ** 2))

    # Load the input _C_0 image
    image_c0 = cv2.imread(input_image_path, cv2.IMREAD_GRAYSCALE)
    if image_c0 is None:
        raise ValueError(f"Error: Unable to load image at {input_image_path}")

    # If bounding boxes are provided, use them directly
    if bbox_list is not None:
        filtered_props = bbox_list
    else:
        # Apply a mean filter (blur the _C_0 image)
        blurred_image = cv2.blur(image_c0, (5, 5))

        # Apply Li thresholding to the blurred image
        li_threshold = threshold_li(blurred_image)
        binary_image = blurred_image > li_threshold
        binary_image = binary_image.astype(np.uint8) * 255

        # Label connected components in the binary image
        labeled_image = skimage_label(binary_image)

        # Get properties of labeled regions
        props = regionprops(labeled_image)

        # Filter out objects that do not meet the minimum size
        filtered_props = [region for region in props if region.area >= min_size_pixels]

        # Randomly select up to 5 objects if more than 5 exist
        if len(filtered_props) > 5:
            filtered_props = random.sample(filtered_props, 5)
            print(f"Randomly selected 5 objects out of {len(props)} total objects.")

    # Get the base filename of the input image
    base_filename_c0 = os.path.basename(input_image_path)

    # Iterate through the selected regions and crop the images
    cropped_count = 0
    for i, region in enumerate(filtered_props):
        # Handle both regionprops objects and bbox tuples
        if hasattr(region, 'bbox'):
            min_row, min_col, max_row, max_col = region.bbox
        else:
            min_row, min_col, max_row, max_col = region

        # Skip objects touching the edges of the image
        if min_row == 0 or min_col == 0 or max_row == image_c0.shape[0] or max_col == image_c0.shape[1]:
            print(f"Skipping object {i + 1} - touching the edge of the image.")
            continue

        # Add padding to the bounding box
        min_row = max(min_row - padding, 0)
        min_col = max(min_col - padding, 0)
        max_row = min(max_row + padding, image_c0.shape[0])
        max_col = min(max_col + padding, image_c0.shape[1])

        # Crop the _C_0 image around the padded bounding box
        cropped_image_c0 = image_c0[min_row:max_row, min_col:max_col]

        # Construct the new filename with _#_ added before _C_0
        cropped_count += 1
        new_filename_c0 = base_filename_c0.replace("_C_0", f"_{cropped_count}_C_0")

        # Save the cropped _C_0 image in the output folder
        output_path_c0 = os.path.join(output_folder, new_filename_c0)
        cv2.imwrite(output_path_c0, cropped_image_c0)
        print(f"Saved cropped _C_0 image: {output_path_c0}")

    # Return the bounding boxes (only if we detected them, not if they were provided)
    if bbox_list is None:
        return [region.bbox for region in filtered_props]
    else:
        return None


def Isolate_Microglia(input_folder, padding=20, min_size_microns=100, bbox_dict=None):
    """
    Processes all images in a folder that end with _C_0.tiff for microglia (single-channel).
    Groups images by their base name (after removing the prefix like 0_, 5_, 10_, 18_, 25_, 35_, 50_, 100_).
    Uses the same crop locations across all images with the same base name.
    Automatically creates an 'Isolated microglia' folder inside input_folder.
    Uses Li thresholding method.
    
    If bbox_dict is provided, uses those bounding boxes for matching filenames.
    Returns a dictionary mapping base filenames to their bounding boxes.
    """
    import os
    from collections import defaultdict

    # Create "Isolated microglia" subfolder
    output_folder = os.path.join(input_folder, "Isolated microglia")
    os.makedirs(output_folder, exist_ok=True)

    # Group files by their base name (without prefix)
    grouped_files = defaultdict(list)
    
    for file_name in os.listdir(input_folder):
        if file_name.endswith("_C_0.tiff"):
            # Extract the base name without the prefix (0_, 5_, 10_, 18_, 25_, 35_, 50_, 100_)
            parts = file_name.split('_', 1)
            if len(parts) == 2 and parts[0] in ['0', '5', '10', '18', '25', '35', '50', '100']:
                base_name = parts[1]  # Everything after the prefix
                grouped_files[base_name].append(file_name)
            else:
                # If no recognized prefix, treat the whole name as base
                grouped_files[file_name].append(file_name)

    # Dictionary to store bounding boxes for each base filename
    local_bbox_dict = {}

    # Process each group
    for base_name, file_list in grouped_files.items():
        print(f"\nProcessing group: {base_name}")
        print(f"Files in group: {file_list}")
        
        # Sort to ensure consistent order (0_, 5_, 10_, 18_, 25_, 35_, 50_, 100_)
        file_list.sort()
        
        # Check if we have bounding boxes from a previous folder
        if bbox_dict is not None and base_name in bbox_dict:
            print(f"Using existing bounding boxes for: {base_name}")
            bbox_list = bbox_dict[base_name]
            
            # Apply the bounding boxes to all images in this group
            for file_name in file_list:
                file_path = os.path.join(input_folder, file_name)
                print(f"Applying same crops to: {file_name}")
                Crop_Microglia(
                    file_path,
                    output_folder,
                    padding=padding,
                    min_size_microns=min_size_microns,
                    bbox_list=bbox_list
                )
            
            # Store the bounding boxes for this base name
            local_bbox_dict[base_name] = bbox_list
        else:
            # Detect new bounding boxes from the first image
            first_file = file_list[0]
            first_file_path = os.path.join(input_folder, first_file)
            
            print(f"Detecting objects in: {first_file}")
            bbox_list = Crop_Microglia(
                first_file_path,
                output_folder,
                padding=padding,
                min_size_microns=min_size_microns,
                bbox_list=None  # Detect bounding boxes
            )
            
            # Store the bounding boxes for this base name
            local_bbox_dict[base_name] = bbox_list
            
            # Apply the same bounding boxes to the remaining images in the group
            for file_name in file_list[1:]:
                file_path = os.path.join(input_folder, file_name)
                print(f"Applying same crops to: {file_name}")
                Crop_Microglia(
                    file_path,
                    output_folder,
                    padding=padding,
                    min_size_microns=min_size_microns,
                    bbox_list=bbox_list
                )
    
    return local_bbox_dict


def Process_All_Subfolders(main_directory, padding=20, min_size_microns=100, reference_folder_name='r_50'):
    """
    Iterates through all subfolders in the main directory and applies Isolate_Microglia
    to each subfolder that contains _C_0.tiff images.
    
    ENSURES THE SAME MICROGLIA ARE CROPPED ACROSS ALL SUBFOLDERS by:
    1. First processing the reference folder (default: 'r_50') to detect microglia locations
    2. Using those same bounding boxes for all other subfolders
    
    Supported radius prefixes: 0, 10, 18, 25, 35, 50
    
    Expected structure:
    main_directory/
    ├── r_0/ (contains _C_0.tiff files directly)
    ├── r_10/ (contains _C_0.tiff files directly)
    ├── r_18/ (contains _C_0.tiff files directly)
    ├── r_25/ (contains _C_0.tiff files directly)
    ├── r_35/ (contains _C_0.tiff files directly)
    └── r_50/ (contains _C_0.tiff files directly)
    
    Parameters:
    - main_directory: Path to the main directory containing r_* subfolders
    - padding: Padding around cropped objects (default: 20)
    - min_size_microns: Minimum object size in microns² (default: 100)
    - reference_folder_name: Name of the subfolder to use as reference (default: 'r_50')
    """
    import os
    
    print(f"Starting to process all subfolders in: {main_directory}\n")
    print(f"Reference folder for detecting microglia: {reference_folder_name}")
    print(f"Supported radii: 0, 10, 18, 25, 35, 50")
    print("=" * 80)
    
    # Counter for tracking processed folders
    processed_count = 0
    skipped_count = 0
    
    # Dictionary to store bounding boxes from the reference folder
    master_bbox_dict = None
    reference_folder_path = None
    
    # Collect all r_* folders (not nested Isolated microglia folders)
    folders_to_process = []
    
    # Only look at direct subfolders of main_directory
    for item in os.listdir(main_directory):
        item_path = os.path.join(main_directory, item)
        
        # Only process directories starting with 'r_'
        if os.path.isdir(item_path) and item.startswith('r_'):
            # Check if this folder contains _C_0.tiff files (not in a subfolder)
            has_c0_files = any(file.endswith("_C_0.tiff") for file in os.listdir(item_path))
            
            if has_c0_files:
                folders_to_process.append(item_path)
                print(f"✓ Found: {item} (contains {sum(1 for f in os.listdir(item_path) if f.endswith('_C_0.tiff'))} images)")
            else:
                print(f"⚠️  Skipping: {item} (no _C_0.tiff files in root)")
    
    if not folders_to_process:
        print("\n❌ No r_* folders with _C_0.tiff files found!")
        print("\nExpected structure:")
        print("  main_directory/")
        print("  ├── r_0/ (with _C_0.tiff files)")
        print("  ├── r_10/ (with _C_0.tiff files)")
        print("  ...")
        return
    
    # Find the reference folder (e.g., r_50)
    reference_folder_path = None
    for folder in folders_to_process:
        folder_name = os.path.basename(folder)
        if reference_folder_name == folder_name or reference_folder_name in folder_name:
            reference_folder_path = folder
            break
    
    if reference_folder_path is None:
        print(f"\n⚠️  WARNING: Reference folder '{reference_folder_name}' not found!")
        print(f"Available folders: {[os.path.basename(f) for f in folders_to_process]}")
        print(f"Defaulting to first folder: {folders_to_process[0]}")
        reference_folder_path = folders_to_process[0]
    else:
        print(f"\n✓ Found reference folder: {os.path.basename(reference_folder_path)}")
    
    # Sort remaining folders (excluding reference)
    other_folders = [f for f in folders_to_process if f != reference_folder_path]
    other_folders.sort()
    
    # Reorder: reference folder first, then all others
    folders_to_process = [reference_folder_path] + other_folders
    
    print(f"\nProcessing order:")
    for i, folder in enumerate(folders_to_process, 1):
        marker = "🔍 [REFERENCE - DETECTING LOCATIONS]" if i == 1 else "📋 [USING REFERENCE LOCATIONS]"
        print(f"  {i}. {os.path.basename(folder)} {marker}")
    print("=" * 80)
    
    # Process each folder
    for i, folder in enumerate(folders_to_process):
        print(f"\n{'=' * 80}")
        print(f"Processing folder {i+1}/{len(folders_to_process)}: {os.path.basename(folder)}")
        print(f"{'=' * 80}")
        
        try:
            if i == 0:
                # First folder (reference): detect bounding boxes
                print(f"🔍 REFERENCE FOLDER ({reference_folder_name}): Detecting microglia locations...")
                master_bbox_dict = Isolate_Microglia(
                    folder, 
                    padding=padding, 
                    min_size_microns=min_size_microns,
                    bbox_dict=None
                )
                print(f"✓ Detected bounding boxes for {len(master_bbox_dict)} unique image groups")
                print(f"✓ These locations will be used for all subsequent folders")
            else:
                # Subsequent folders: use the same bounding boxes
                print(f"📋 Using bounding boxes from reference folder: {os.path.basename(reference_folder_path)}")
                Isolate_Microglia(
                    folder, 
                    padding=padding, 
                    min_size_microns=min_size_microns,
                    bbox_dict=master_bbox_dict
                )
            
            processed_count += 1
            print(f"✓ Successfully processed: {os.path.basename(folder)}")
            
        except Exception as e:
            print(f"✗ Error processing {os.path.basename(folder)}: {str(e)}")
            import traceback
            traceback.print_exc()
            skipped_count += 1
    
    # Summary
    print(f"\n{'=' * 80}")
    print(f"PROCESSING COMPLETE")
    print(f"{'=' * 80}")
    print(f"Reference folder used: {os.path.basename(reference_folder_path)}")
    print(f"Total folders processed: {processed_count}")
    print(f"Total folders skipped: {skipped_count}")
    if master_bbox_dict:
        print(f"Total unique image groups: {len(master_bbox_dict)}")
        print(f"Master bounding boxes detected from: {reference_folder_path}")
    print(f"Isolated microglia saved to: [folder]/Isolated microglia/")
    print(f"{'=' * 80}\n")


# ================================================================================
# USAGE
# ================================================================================

# Process all subfolders, using r_50 as the reference for detecting microglia
Process_All_Subfolders(folder_path, padding=20, min_size_microns=250, reference_folder_name='r_50')

# Or use a different reference folder:
# Process_All_Subfolders(folder_path, padding=20, min_size_microns=250, reference_folder_name='r_35')

In [ ]:
def sync_isolated_microglia_folders(main_directory, reference_folder_name='r_50', dry_run=True):
    """
    Synchronizes Isolated Microglia folders across all radius subfolders to match
    the reference folder (default: r_50). Deletes images from other folders if they
    don't exist in the reference folder.
    
    This ensures all radius folders contain exactly the same set of images.
    
    Directory structure expected:
    main_directory/
    ├── r_0/
    │   └── Isolated Microglia/
    │       ├── image1_1_C_0.tiff
    │       └── image1_2_C_0.tiff
    ├── r_50/
    │   └── Isolated Microglia/
    │       └── image1_1_C_0.tiff  (image1_2 was deleted)
    └── ...
    
    Parameters:
    -----------
    main_directory : str
        Path to the main directory containing r_X subfolders
    reference_folder_name : str, optional
        Name of the reference folder to use as the standard (default: 'r_50')
    dry_run : bool, optional
        If True, only shows what would be deleted without actually deleting.
        If False, performs the actual deletion. (default: True for safety)
    
    Returns:
    --------
    summary : dict
        Dictionary containing:
        - 'reference_folder': Path to reference folder
        - 'reference_images': List of images in reference folder
        - 'folders_processed': Number of folders processed
        - 'images_deleted': Total number of images deleted
        - 'deletion_details': Dict mapping folder names to lists of deleted images
    
    Example:
    --------
    # Preview what would be deleted (safe mode)
    summary = sync_isolated_microglia_folders(
        main_directory='/path/to/Background_Subtracted_Images',
        reference_folder_name='r_50',
        dry_run=True
    )
    
    # Actually perform the deletion
    summary = sync_isolated_microglia_folders(
        main_directory='/path/to/Background_Subtracted_Images',
        reference_folder_name='r_50',
        dry_run=False
    )
    """
    import os
    
    print(f"\n{'='*80}")
    print(f"SYNCHRONIZING ISOLATED MICROGLIA FOLDERS")
    print(f"{'='*80}\n")
    print(f"Main directory: {main_directory}")
    print(f"Reference folder: {reference_folder_name}")
    print(f"Mode: {'DRY RUN (preview only)' if dry_run else 'DELETION MODE (will delete files)'}")
    print()
    
    # Verify main directory exists
    if not os.path.exists(main_directory):
        print(f"❌ ERROR: Main directory not found: {main_directory}")
        return None
    
    # Find the reference folder
    reference_folder_path = None
    reference_isolated_path = None
    
    for item in os.listdir(main_directory):
        item_path = os.path.join(main_directory, item)
        if os.path.isdir(item_path) and reference_folder_name in item:
            reference_folder_path = item_path
            reference_isolated_path = os.path.join(item_path, 'Isolated Microglia')
            break
    
    if reference_folder_path is None:
        print(f"❌ ERROR: Reference folder '{reference_folder_name}' not found in {main_directory}")
        return None
    
    if not os.path.exists(reference_isolated_path):
        print(f"❌ ERROR: 'Isolated Microglia' folder not found in {reference_folder_path}")
        return None
    
    print(f"✓ Found reference folder: {reference_folder_path}")
    print(f"✓ Reference Isolated Microglia: {reference_isolated_path}")
    print()
    
    # Get list of images in reference folder
    try:
        reference_images = set(os.listdir(reference_isolated_path))
        # Filter to only image files
        reference_images = {f for f in reference_images 
                           if f.lower().endswith(('.tif', '.tiff', '.png', '.jpg'))}
    except Exception as e:
        print(f"❌ ERROR: Unable to read reference folder: {e}")
        return None
    
    print(f"Reference folder contains {len(reference_images)} image(s)")
    print(f"{'─'*80}")
    print(f"Reference images:")
    for img in sorted(reference_images):
        print(f"  ✓ {img}")
    print(f"{'─'*80}\n")
    
    # Track statistics
    folders_processed = 0
    total_images_deleted = 0
    deletion_details = {}
    
    # Find all other radius folders
    radius_folders = []
    for item in os.listdir(main_directory):
        item_path = os.path.join(main_directory, item)
        if os.path.isdir(item_path) and item.startswith('r_') and item != os.path.basename(reference_folder_path):
            isolated_path = os.path.join(item_path, 'Isolated Microglia')
            if os.path.exists(isolated_path):
                radius_folders.append((item, item_path, isolated_path))
    
    if not radius_folders:
        print("⚠️  No other radius folders found to synchronize")
        return {
            'reference_folder': reference_folder_path,
            'reference_images': list(reference_images),
            'folders_processed': 0,
            'images_deleted': 0,
            'deletion_details': {}
        }
    
    print(f"Found {len(radius_folders)} other radius folder(s) to synchronize:")
    for folder_name, _, _ in radius_folders:
        print(f"  - {folder_name}")
    print()
    
    # Process each folder
    for folder_name, folder_path, isolated_path in sorted(radius_folders):
        print(f"{'='*80}")
        print(f"Processing: {folder_name}")
        print(f"{'='*80}")
        
        try:
            # Get list of images in this folder
            current_images = set(os.listdir(isolated_path))
            current_images = {f for f in current_images 
                            if f.lower().endswith(('.tif', '.tiff', '.png', '.jpg'))}
            
            print(f"Current folder contains {len(current_images)} image(s)")
            
            # Find images that exist here but not in reference
            images_to_delete = current_images - reference_images
            
            if not images_to_delete:
                print(f"✓ Already synchronized - no images to delete")
                folders_processed += 1
            else:
                print(f"⚠️  Found {len(images_to_delete)} image(s) to delete:")
                
                deleted_images = []
                
                for img in sorted(images_to_delete):
                    img_path = os.path.join(isolated_path, img)
                    
                    if dry_run:
                        print(f"  [DRY RUN] Would delete: {img}")
                        deleted_images.append(img)
                    else:
                        try:
                            os.remove(img_path)
                            print(f"  ✗ Deleted: {img}")
                            deleted_images.append(img)
                            total_images_deleted += 1
                        except Exception as e:
                            print(f"  ❌ Error deleting {img}: {e}")
                
                deletion_details[folder_name] = deleted_images
                folders_processed += 1
            
            print()
            
        except Exception as e:
            print(f"❌ Error processing {folder_name}: {e}")
            import traceback
            traceback.print_exc()
            print()
    
    # Summary
    print(f"{'='*80}")
    print(f"SYNCHRONIZATION {'PREVIEW' if dry_run else 'COMPLETE'}")
    print(f"{'='*80}")
    print(f"Reference folder: {os.path.basename(reference_folder_path)}")
    print(f"Reference images: {len(reference_images)}")
    print(f"Folders processed: {folders_processed}")
    
    if dry_run:
        print(f"Images that WOULD BE deleted: {sum(len(imgs) for imgs in deletion_details.values())}")
    else:
        print(f"Images deleted: {total_images_deleted}")
    
    if deletion_details:
        print(f"\nDeletion breakdown:")
        for folder, images in sorted(deletion_details.items()):
            print(f"  {folder}: {len(images)} image(s)")
            for img in sorted(images)[:5]:  # Show first 5
                print(f"    - {img}")
            if len(images) > 5:
                print(f"    ... and {len(images) - 5} more")
    else:
        print(f"\n✓ All folders already synchronized!")
    
    if dry_run:
        print(f"\n⚠️  DRY RUN MODE - No files were actually deleted")
        print(f"   Run with dry_run=False to perform actual deletion")
    
    print(f"{'='*80}\n")
    
    return {
        'reference_folder': reference_folder_path,
        'reference_images': list(reference_images),
        'folders_processed': folders_processed,
        'images_deleted': total_images_deleted if not dry_run else 0,
        'deletion_details': deletion_details,
        'dry_run': dry_run
    }


# ================================================================================
# EXAMPLE USAGE
# ================================================================================

# # Example 1: Preview what would be deleted (SAFE - recommended first)
# print("=" * 80)
# print("STEP 1: PREVIEW MODE (Safe - just shows what would be deleted)")
# print("=" * 80)
# summary = sync_isolated_microglia_folders(
#     main_directory='/Users/allisonpickle/Desktop/Image Processing Pipeline Analysis/All Microglia Copied/Background_Subtracted_Images',
#     reference_folder_name='r_50',
#     dry_run=True  # Safe mode - just preview
# )

# Example 2: Actually delete the files (after reviewing the preview)
# Uncomment the lines below after reviewing the preview above
print("\n" + "=" * 80)
print("STEP 2: DELETION MODE (Will actually delete files)")
print("=" * 80)
summary = sync_isolated_microglia_folders(
    main_directory='/Users/allisonpickle/Desktop/Image Processing Pipeline Analysis/All Microglia Copied/Background_Subtracted_Images',
    reference_folder_name='r_50',
    dry_run=False  # Actually delete
)

# Example 3: Use a different reference folder (e.g., r_35)
# summary = sync_isolated_microglia_folders(
#     main_directory='/path/to/Background_Subtracted_Images',
#     reference_folder_name='r_35',
#     dry_run=True
# )

# Access summary information
if summary:
    print("\n" + "=" * 80)
    print("SUMMARY DETAILS")
    print("=" * 80)
    print(f"Reference folder: {summary['reference_folder']}")
    print(f"Reference contains: {len(summary['reference_images'])} images")
    print(f"Folders processed: {summary['folders_processed']}")
    
    if summary['dry_run']:
        total_would_delete = sum(len(imgs) for imgs in summary['deletion_details'].values())
        print(f"Would delete: {total_would_delete} images")
    else:
        print(f"Deleted: {summary['images_deleted']} images")
    
    if summary['deletion_details']:
        print(f"\nAffected folders: {len(summary['deletion_details'])}")
        for folder, images in summary['deletion_details'].items():
            print(f"  - {folder}: {len(images)} images")
    print("=" * 80)

In [ ]:
Measure_Microglia_Morphology(folder_path)

In [ ]:
def Measure_Microglia_Morphology_Single_Radius_Optimized(colorized_image_path, 
                                                         selected_images=None, 
                                                         contour_sample=None,
                                                         contour_dpi=150):
    """
    Measure morphology of microglia in a single radius folder with optimized contour saving.
    
    Parameters:
    - colorized_image_path: Path to the folder containing microglia images
    - selected_images: List of specific image filenames to process (if None, processes all)
    - contour_sample: Set of image filenames to save contours for (None = no contours)
    - contour_dpi: DPI for saved contour images (default: 150)
    
    Returns:
    - DataFrame with results including radius information
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt
    from skimage import measure, io
    from skimage.filters import threshold_mean
    from skimage.morphology import binary_closing, disk, convex_hull_image
    from skimage.measure import regionprops, label, perimeter
    import pandas as pd
    import math
    from scipy.ndimage import center_of_mass

    def calculate_inertia_tensor(binary_image):
        """Calculate the inertia tensor of a binary image based on its center of mass."""
        com = center_of_mass(binary_image)
        y_indices, x_indices = np.indices(binary_image.shape)
        x_distances = x_indices - com[1]
        y_distances = y_indices - com[0]
        Ixx = np.sum(binary_image * y_distances**2)
        Iyy = np.sum(binary_image * x_distances**2)
        Ixy = -np.sum(binary_image * x_distances * y_distances)
        return np.array([[Ixx, Ixy], [Ixy, Iyy]])

    def calculate_aspect_ratio(eigenvalues):
        """Calculate the aspect ratio as the square root of the ratio of the largest to the smallest eigenvalue."""
        return np.sqrt(np.max(eigenvalues) / np.min(eigenvalues))

    def get_radius_from_path(path):
        """Extract radius from parent folder name (e.g., r_50 -> 50)"""
        parent = os.path.basename(os.path.dirname(path))
        if parent.startswith('r_'):
            return parent.split('_')[1]
        return 'Unknown'

    # Get radius from folder path
    radius = get_radius_from_path(colorized_image_path)
    
    # Set scale for 20x Objective
    scale = 0.335  # microns per pixel

    # Create a "Contours" folder if we're saving any contours
    if contour_sample:
        contours_folder = os.path.join(colorized_image_path, "Contours")
        os.makedirs(contours_folder, exist_ok=True)

    # Get list of image files
    all_image_files = [file for file in os.listdir(colorized_image_path) 
                       if file.endswith('_C_0.tif') or file.endswith('_C_0.tiff')]
    
    # Use selected images if provided, otherwise use all
    if selected_images is not None:
        image_files = [f for f in all_image_files if f in selected_images]
    else:
        image_files = all_image_files

    # Process each image
    results = []
    for image_file in image_files:
        image_path = os.path.join(colorized_image_path, image_file)
        image = io.imread(image_path, as_gray=True)

        # Apply Mean thresholding
        thresh = threshold_mean(image)
        binary_image = image > thresh
        binary_image = binary_closing(binary_image, disk(3))

        # Label connected components
        labeled_image = label(binary_image)
        properties = regionprops(labeled_image)

        if len(properties) == 0:
            print(f"Warning: No objects found in {image_file}")
            continue

        # Find largest region
        largest_region = max(properties, key=lambda x: x.area)

        # Calculate convex hull
        convex_hull = convex_hull_image(labeled_image == largest_region.label)
        convex_hull_perimeter_pixels = perimeter(convex_hull)
        convex_hull_perimeter_microns = convex_hull_perimeter_pixels * scale
        convex_hull_area_pixels = np.sum(convex_hull)
        convex_hull_area_microns = convex_hull_area_pixels * (scale ** 2)

        # Calculate measurements
        area_pixels = largest_region.area
        perimeter_pixels = largest_region.perimeter
        area_microns = area_pixels * (scale ** 2)
        perimeter_microns = perimeter_pixels * scale

        # Calculate circularity
        circularity = (4 * math.pi * area_microns) / (perimeter_microns ** 2) if perimeter_microns > 0 else 0

        # Calculate aspect ratio
        binary_region = labeled_image == largest_region.label
        inertia_tensor = calculate_inertia_tensor(binary_region)
        eigenvalues, eigenvectors = np.linalg.eigh(inertia_tensor)
        aspect_ratio = calculate_aspect_ratio(eigenvalues)

        # Calculate roughness
        roughness = perimeter_microns / convex_hull_perimeter_microns if convex_hull_perimeter_microns > 0 else 0

        # Extract base name (remove radius prefix for matching across radii)
        parts = image_file.replace('_C_0.tif', '').replace('_C_0.tiff', '').split('_')
        if parts[0] in ['0', '10', '18', '25', '35', '50']:
            base_name = "_".join(parts[1:])  # Remove radius prefix
        else:
            base_name = "_".join(parts)

        # Append results
        results.append({
            "File": image_file,
            "Base Name": base_name,
            "Radius": radius,
            "Area (µm²)": area_microns,
            "Perimeter (µm)": perimeter_microns,
            "Convex Hull Area (µm²)": convex_hull_area_microns,
            "Convex Hull Perimeter (µm)": convex_hull_perimeter_microns,
            "Circularity": circularity,
            "Aspect Ratio": aspect_ratio,
            "Roughness": roughness
        })

        # Save contour ONLY if in sample
        if contour_sample and image_file in contour_sample:
            fig, ax = plt.subplots(figsize=(10, 10))
            ax.imshow(image, cmap='gray')
            
            # Find and plot contours
            contours = measure.find_contours(labeled_image == largest_region.label, 0.5)
            for contour in contours:
                ax.plot(contour[:, 1], contour[:, 0], linewidth=2, color='red')
            
            # OPTIMIZED: Use scatter instead of loop for convex hull
            convex_hull_coords = np.argwhere(convex_hull)
            ax.scatter(convex_hull_coords[:, 1], convex_hull_coords[:, 0], 
                      s=1, c='green', marker='o', alpha=0.5)
            
            ax.axis('off')
            
            # Create output filename as PNG (removes _C_0.tif/.tiff and adds _contour.png)
            base_filename = image_file.replace('_C_0.tif', '').replace('_C_0.tiff', '')
            output_filename = f"{base_filename}_contour.png"
            output_path = os.path.join(contours_folder, output_filename)
            
            # Save as PNG explicitly
            plt.savefig(output_path, dpi=contour_dpi, bbox_inches='tight', pad_inches=0, format='png')
            plt.close(fig)
    
    return pd.DataFrame(results)


def Process_All_Radii_All_Images(main_directory, save_contours_sample=20, contour_dpi=150):
    """
    Process ALL microglia images across all radius folders.
    
    Workflow:
    1. Find all r_* folders with "Isolated Microglia" subfolders
    2. Process ALL images in each folder (calculations for all)
    3. Save contours only for a random sample (default: 20 per radius)
    4. Combine all results into a single Excel file with pivot tables (radius as columns)
    
    Parameters:
    - main_directory: Path to directory containing r_* subfolders
    - save_contours_sample: Number of random images per radius to save contours for (default: 20)
                           Set to None for all images, or 0 for no contours
    - contour_dpi: DPI for saved contour images (default: 150, use 300 for publication)
    
    Expected structure:
    main_directory/
    ├── r_0/Isolated Microglia/
    ├── r_10/Isolated Microglia/
    ├── r_18/Isolated Microglia/
    ├── r_25/Isolated Microglia/
    ├── r_35/Isolated Microglia/
    └── r_50/Isolated Microglia/
    
    Output contours:
    main_directory/
    ├── r_0/Isolated Microglia/Contours/
    │   ├── 0_image1_1_contour.png
    │   ├── 0_image1_2_contour.png
    │   └── ...
    ├── r_10/Isolated Microglia/Contours/
    │   ├── 10_image1_1_contour.png
    │   └── ...
    """
    import os
    import pandas as pd
    import random
    
    print("=" * 80)
    print("PROCESSING ALL MICROGLIA ACROSS ALL RADII")
    print("=" * 80)
    print(f"Main directory: {main_directory}")
    if save_contours_sample:
        print(f"Contours will be saved for: {save_contours_sample} random images per radius")
        print(f"Contour format: PNG with '_contour' suffix")
    else:
        print(f"Contours: DISABLED (calculations only)")
    print(f"Contour DPI: {contour_dpi}\n")
    
    # Define expected radii
    expected_radii = ['0', '10', '18', '25', '35', '50']
    
    # Find all radius folders with "Isolated Microglia" subfolder
    radius_folders = {}
    
    for radius in expected_radii:
        folder_name = f'r_{radius}'
        folder_path = os.path.join(main_directory, folder_name)
        isolated_path = os.path.join(folder_path, 'Isolated Microglia')
        
        if os.path.exists(isolated_path):
            # Count images
            image_count = sum(1 for f in os.listdir(isolated_path) 
                            if f.endswith('_C_0.tif') or f.endswith('_C_0.tiff'))
            radius_folders[radius] = isolated_path
            print(f"✓ Found: {folder_name}/Isolated Microglia ({image_count} images)")
        else:
            print(f"✗ Missing: {folder_name}/Isolated Microglia")
    
    if not radius_folders:
        print("\n❌ No radius folders found!")
        return
    
    print(f"\n✓ Found {len(radius_folders)} radius folder(s)")
    
    # Process each radius folder
    print("\n" + "=" * 80)
    print("PROCESSING ALL FOLDERS")
    print("=" * 80)
    
    all_results = []
    
    for radius in sorted(radius_folders.keys()):
        folder_path = radius_folders[radius]
        
        print(f"\nProcessing r_{radius}...")
        
        # Get all images
        all_images = [f for f in os.listdir(folder_path) 
                     if f.endswith('_C_0.tif') or f.endswith('_C_0.tiff')]
        
        # Randomly select images for contour saving
        if save_contours_sample and save_contours_sample > 0:
            contour_sample = set(random.sample(all_images, 
                                              min(save_contours_sample, len(all_images))))
        else:
            contour_sample = None
        
        try:
            # Process with selective contour saving
            results_df = Measure_Microglia_Morphology_Single_Radius_Optimized(
                folder_path,
                selected_images=None,  # Process ALL images
                contour_sample=contour_sample,
                contour_dpi=contour_dpi
            )
            all_results.append(results_df)
            
            if contour_sample:
                print(f"  ✓ Processed {len(results_df)} images ({len(contour_sample)} contours saved as PNG)")
            else:
                print(f"  ✓ Processed {len(results_df)} images (no contours)")
                
        except Exception as e:
            print(f"  ✗ Error: {str(e)}")
            import traceback
            traceback.print_exc()
    
    # Combine results
    if not all_results:
        print("\n❌ No results to combine!")
        return
    
    print("\n" + "=" * 80)
    print("COMBINING RESULTS AND CREATING EXCEL FILE")
    print("=" * 80)
    
    combined_df = pd.concat(all_results, ignore_index=True)
    
    print(f"\nTotal rows: {len(combined_df)}")
    print(f"Unique microglia: {combined_df['Base Name'].nunique()}")
    print(f"Radii processed: {sorted(combined_df['Radius'].unique())}")
    
    # Count images per radius
    print(f"\nImages per radius:")
    for radius in sorted(combined_df['Radius'].unique()):
        count = len(combined_df[combined_df['Radius'] == radius])
        print(f"  r_{radius}: {count} images")
    
    # Create Excel file with multiple sheets
    excel_path = os.path.join(main_directory, "Combined_Morphology_Results_All_Images.xlsx")
    
    measurements = ['Area (µm²)', 'Perimeter (µm)', 'Convex Hull Area (µm²)', 
                   'Convex Hull Perimeter (µm)', 'Circularity', 'Aspect Ratio', 'Roughness']
    
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        # Sheet 1: All data with all columns (long format)
        combined_df_sorted = combined_df.sort_values(['Base Name', 'Radius']).reset_index(drop=True)
        combined_df_sorted.to_excel(writer, sheet_name='All_Data', index=False)
        
        # Sheets 2-8: Pivot tables with radius as columns (wide format)
        for measurement in measurements:
            # Create pivot table: Base Name as rows, Radius as columns
            pivot_df = combined_df.pivot_table(
                index='Base Name',
                columns='Radius',
                values=measurement,
                aggfunc='first'  # Use 'first' since each Base Name should have one value per radius
            )
            
            # Reorder columns to match radius order
            radius_order = ['0', '10', '18', '25', '35', '50']
            existing_radii = [r for r in radius_order if r in pivot_df.columns]
            pivot_df = pivot_df[existing_radii]
            
            # Rename columns to include 'r_' prefix for clarity
            pivot_df.columns = [f'r_{col}' for col in pivot_df.columns]
            
            # Reset index to make Base Name a column
            pivot_df = pivot_df.reset_index()
            
            # Clean sheet name (Excel has 31 character limit)
            sheet_name = measurement.replace(' (µm²)', '').replace(' (µm)', '').replace(' ', '_')
            sheet_name = sheet_name[:31]
            
            # Write to Excel
            pivot_df.to_excel(writer, sheet_name=sheet_name, index=False)
    
    print(f"\n✓ Results exported to: {excel_path}")
    print(f"\nExcel sheets created:")
    print(f"  1. All_Data (long format: all columns, {len(combined_df)} rows)")
    for i, measurement in enumerate(measurements, 2):
        sheet_name = measurement.replace(' (µm²)', '').replace(' (µm)', '').replace(' ', '_')[:31]
        unique_base_names = combined_df['Base Name'].nunique()
        print(f"  {i}. {sheet_name} (wide format: Base Name | r_0 | r_10 | r_18 | r_25 | r_35 | r_50, {unique_base_names} rows)")
    
    # Summary
    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"Total images processed: {len(combined_df)}")
    print(f"Total unique microglia (Base Names): {combined_df['Base Name'].nunique()}")
    print(f"Total radius folders: {len(radius_folders)}")
    if save_contours_sample:
        print(f"Contours saved: {save_contours_sample} per radius ({save_contours_sample * len(radius_folders)} total)")
        print(f"Contour format: PNG files with '_contour' suffix")
        print(f"Contour DPI: {contour_dpi}")
        print(f"Example: 0_image1_1_C_0.tiff → 0_image1_1_contour.png")
    else:
        print(f"Contours: Not saved (calculations only)")
    print(f"Excel file: {os.path.basename(excel_path)}")
    print(f"Format: Wide format with radius as columns (r_0, r_10, r_18, r_25, r_35, r_50)")
    print("=" * 80 + "\n")
    
    return combined_df


# ================================================================================
# USAGE OPTIONS
# ================================================================================

# OPTION 1: Fast (5-10 minutes) - 20 contours per radius for validation
# Process_All_Radii_All_Images(folder_path, save_contours_sample=20, contour_dpi=150)

# OPTION 2: Very fast (2-3 minutes) - no contours, just calculations
Process_All_Radii_All_Images(folder_path, save_contours_sample=0)

# OPTION 3: Slow (1-2 hours) - all contours at medium quality
# Process_All_Radii_All_Images(folder_path, save_contours_sample=None, contour_dpi=150)

# OPTION 4: Very slow (3-4 hours) - all contours at publication quality
# Process_All_Radii_All_Images(folder_path, save_contours_sample=None, contour_dpi=300)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import defaultdict

def create_composite_images_for_radii(main_directory,
                                      radii=('0', '10', '18', '25', '35', '50'),
                                      image_extensions=('.tif', '.tiff', '.png', '.jpg'),
                                      figsize_per_image=(3, 3),
                                      dpi=150):
    """
    Iterate through r_0, r_10, r_18, r_25, r_35, r_50 subfolders in main_directory and
    compile images of the same base name into composite images with columns labeled by radius.
    Creates composites for both:
      - Isolated Microglia: main_directory/r_#/Isolated Microglia
      - Contours: main_directory/r_#/Isolated Microglia/Contours
    Column labels are the radius (r = #). The image file name is NOT drawn in the figure.
    Output files are saved under:
      - main_directory/Radius_Comparisons/Isolated_Microglia
      - main_directory/Radius_Comparisons/Contours
    """
    def collect_by_basename(folder_paths):
        # Map: base_name -> {radius: path}
        by_name = defaultdict(dict)
        for radius, folder in folder_paths.items():
            if not os.path.exists(folder):
                continue
            try:
                files = [f for f in os.listdir(folder)
                         if any(f.lower().endswith(ext) for ext in image_extensions)]
            except Exception:
                continue
            for f in files:
                base = os.path.splitext(f)[0]
                by_name[base][radius] = os.path.join(folder, f)
        return by_name

    def ensure_output_dirs(root):
        out_root = os.path.join(root, 'Radius_Comparisons')
        iso_out = os.path.join(out_root, 'Isolated_Microglia')
        con_out = os.path.join(out_root, 'Contours')
        os.makedirs(iso_out, exist_ok=True)
        os.makedirs(con_out, exist_ok=True)
        return iso_out, con_out

    def make_composites(by_name, label, output_dir):
        # Only include images present in ALL requested radii
        complete = {name: paths for name, paths in by_name.items() if len(paths) == len(radii)}
        print(f"{label}: {len(complete)} composite(s) to create (complete across radii)")
        count = 0
        for base_name, paths in sorted(complete.items()):
            ncols = len(radii)
            fig, axes = plt.subplots(1, ncols,
                                     figsize=(figsize_per_image[0] * ncols, figsize_per_image[1]))
            if ncols == 1:
                axes = [axes]
            # Render each radius column
            for ax_idx, radius in enumerate(radii):
                ax = axes[ax_idx]
                img_path = paths[radius]
                try:
                    img = Image.open(img_path)
                    arr = np.array(img)
                    if arr.ndim == 2:
                        ax.imshow(arr, cmap='gray')
                    else:
                        ax.imshow(arr)
                except Exception as e:
                    ax.text(0.5, 0.5, f'Error\nr_{radius}', ha='center', va='center', color='red')
                ax.set_title(f'r = {radius}', fontsize=12, fontweight='bold', pad=8)
                ax.axis('off')
            plt.subplots_adjust(wspace=0.05, hspace=0, left=0.02, right=0.98, top=0.88, bottom=0.02)
            # Save without drawing file name text in the figure
            # Use base_name for the output filename, but no filename text rendered
            safe_name = base_name  # no drawing of filename on canvas
            out_path = os.path.join(output_dir, f"{safe_name}.png")
            plt.savefig(out_path, dpi=dpi, bbox_inches='tight', facecolor='white', edgecolor='none')
            plt.close(fig)
            count += 1
        return count

    # Prepare folder maps
    isolated_paths = {}
    contour_paths = {}
    for r in radii:
        isolated_paths[r] = os.path.join(main_directory, f"r_{r}", "Isolated Microglia")
        contour_paths[r] = os.path.join(main_directory, f"r_{r}", "Isolated Microglia", "Contours")

    iso_out, con_out = ensure_output_dirs(main_directory)

    print("Collecting Isolated Microglia images...")
    iso_by_name = collect_by_basename(isolated_paths)

    print("Collecting Contour images...")
    con_by_name = collect_by_basename(contour_paths)

    print("Creating composites (Isolated Microglia)...")
    iso_count = make_composites(iso_by_name, "Isolated Microglia", iso_out)

    print("Creating composites (Contours)...")
    con_count = make_composites(con_by_name, "Contours", con_out)

    print(f"Done. Isolated composites: {iso_count}, Contour composites: {con_count}")
    return {
        "isolated_composites": iso_count,
        "contour_composites": con_count,
        "output_directories": {
            "isolated": iso_out,
            "contours": con_out
        }
    }

# Example usage:
main_dir = '/Users/allisonpickle/Desktop/Image Processing Pipeline Analysis/All Microglia Copied/Background_Subtracted_Images'
create_composite_images_for_radii(main_dir)

In [ ]:
def Analyze_Background_Subtraction_Stats(
    excel_path,
    sheet_name=None,
    output_dir=None,
    expected_radii=('0', '10', '18', '25', '35', '50'),
    reference_radius='0',
    alpha=0.05,
    filter_treatment='5ugLPS'  # NEW: Filter for specific treatment
):
    """
    Performs repeated measures ANOVA on background subtraction data from Excel.
    
    Expected Excel structure:
    Column 0: Filename (e.g., "240623_Tri_5ugLPS_10_20_1_C_0.tiff")
    Subsequent columns: r_0, r_10, r_18, r_25, r_35, r_50
    
    Extracts biological replicate from filename date and performs:
    - Repeated measures ANOVA across radii
    - Pairwise t-tests with Bonferroni correction
    - Generates plots and saves statistics
    - Plots individual microglia trajectories across radii with significance markers
    
    Parameters:
    - excel_path: Path to Excel file
    - sheet_name: Sheet name (None = first sheet)
    - output_dir: Directory for outputs (None = same as Excel)
    - expected_radii: Tuple of radius values
    - reference_radius: Reference radius for comparisons (default: '0')
    - alpha: Significance level (default: 0.05)
    - filter_treatment: Only process images with this string in filename (default: '5ugLPS')
    """
    import os
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy import stats
    from collections import defaultdict
    
    print(f"Reading Excel file: {excel_path}")
    if filter_treatment:
        print(f"Filtering for images containing: '{filter_treatment}'")
    print("=" * 80)
    
    # Load Excel
    if sheet_name is None:
        df = pd.read_excel(excel_path)
        sheet_name = pd.ExcelFile(excel_path).sheet_names[0]
    else:
        df = pd.read_excel(excel_path, sheet_name=sheet_name)
    
    print(f"Sheet: {sheet_name}")
    print(f"Total rows in Excel: {len(df)}")
    
    # Get headers
    headers = list(df.columns)
    filename_col = headers[0]
    
    # Find radius columns
    radius_columns = {}
    for i, col in enumerate(headers[1:], 1):
        col_name = str(col).strip().lower()
        if col_name.startswith('r_'):
            radius = col_name[2:]
            if radius in expected_radii:
                radius_columns[radius] = col
    
    print(f"Found {len(radius_columns)} radius columns: {list(radius_columns.keys())}")
    
    # Set output directory
    if output_dir is None:
        output_dir = os.path.join(os.path.dirname(excel_path), 
                                   os.path.splitext(os.path.basename(excel_path))[0] + "_Statistics")
    os.makedirs(output_dir, exist_ok=True)
    
    # Group by biological replicate (extract date from filename)
    bio_rep_data = defaultdict(lambda: defaultdict(list))
    
    filtered_count = 0
    total_count = 0
    
    for idx in range(len(df)):
        filename = str(df.iloc[idx][filename_col]).strip()
        if pd.isna(filename) or filename == '':
            continue
        
        total_count += 1
        
        # Filter by treatment if specified
        if filter_treatment and filter_treatment not in filename:
            continue
        
        filtered_count += 1
        
        # Extract date (first underscore-separated token)
        parts = filename.split('_')
        bio_rep = parts[0] if len(parts) > 0 else 'Unknown'
        
        # Collect data for each radius
        for radius, col in radius_columns.items():
            value = df.iloc[idx][col]
            if not pd.isna(value):
                bio_rep_data[bio_rep][radius].append(value)
    
    print(f"\nTotal rows processed: {total_count}")
    if filter_treatment:
        print(f"Rows matching '{filter_treatment}': {filtered_count}")
    
    print(f"\nBiological replicates found: {len(bio_rep_data)}")
    for bio_rep in sorted(bio_rep_data.keys()):
        n_microglia = len(bio_rep_data[bio_rep][list(radius_columns.keys())[0]])
        print(f"  {bio_rep}: {n_microglia} microglia")
    
    print("=" * 80)
    
    # Process each biological replicate
    for bio_rep in sorted(bio_rep_data.keys()):
        print(f"\n{'=' * 80}")
        print(f"Processing Biological Replicate: {bio_rep}")
        print("=" * 80)
        
        # Organize data
        data_dict = bio_rep_data[bio_rep]
        n_microglia = len(data_dict[list(radius_columns.keys())[0]])
        print(f"Number of microglia in Bio Rep {bio_rep}: {n_microglia}")
        
        # Descriptive statistics
        print("\nDESCRIPTIVE STATISTICS:")
        print("-" * 80)
        
        stats_summary = []
        for radius in expected_radii:
            if radius in data_dict:
                values = np.array(data_dict[radius])
                stats_summary.append({
                    'Radius': radius,
                    'Mean': np.mean(values),
                    'SD': np.std(values, ddof=1),
                    'Min': np.min(values),
                    'Max': np.max(values)
                })
                print(f"Radius {radius:>2}: Mean = {np.mean(values):10.4f}, SD = {np.std(values, ddof=1):10.4f}, "
                      f"Min = {np.min(values):10.4f}, Max = {np.max(values):10.4f}")
        
        # Repeated measures ANOVA
        print("\n" + "=" * 80)
        print("STATISTICAL ANALYSIS RESULTS")
        print("=" * 80)
        
        # Prepare data for ANOVA
        radius_list = [r for r in expected_radii if r in data_dict]
        data_matrix = np.array([data_dict[r] for r in radius_list]).T
        
        # Perform repeated measures ANOVA
        F_stat, p_value = stats.f_oneway(*[data_dict[r] for r in radius_list])
        
        print("Repeated Measures ANOVA:")
        print("-" * 80)
        df_between = len(radius_list) - 1
        df_within = n_microglia * len(radius_list) - len(radius_list)
        print(f"F({df_between}, {df_within}) = {F_stat:.4f}")
        print(f"p-value: {p_value:.2e}")
        
        if p_value < 0.001:
            interpretation = "HIGHLY SIGNIFICANT (p < 0.001) ***"
        elif p_value < 0.01:
            interpretation = "SIGNIFICANT (p < 0.01) **"
        elif p_value < 0.05:
            interpretation = "SIGNIFICANT (p < 0.05) *"
        else:
            interpretation = "NOT SIGNIFICANT (p ≥ 0.05) ns"
        
        print(f"Interpretation: {interpretation}")
        
        # Pairwise t-tests with Bonferroni correction
        print(f"\nPairwise paired t-tests vs r_{reference_radius} (reference):")
        n_comparisons = len(radius_list) - 1
        bonferroni_alpha = alpha / n_comparisons
        print(f"Bonferroni correction applied for {n_comparisons} comparisons")
        print("-" * 80)
        
        pairwise_results = []
        ref_data = data_dict[reference_radius]
        
        for radius in radius_list:
            if radius == reference_radius:
                continue
            
            test_data = data_dict[radius]
            t_stat, p_val = stats.ttest_rel(ref_data, test_data)
            
            # Significance stars
            if p_val < 0.001:
                sig = "***"
            elif p_val < 0.01:
                sig = "**"
            elif p_val < bonferroni_alpha:
                sig = "*"
            else:
                sig = "ns"
            
            pairwise_results.append({
                'Comparison': f"r_{reference_radius} vs r_{radius}",
                'Radius': radius,
                't': t_stat,
                'p': p_val,
                'sig': sig
            })
            
            print(f"r_{reference_radius} vs r_{radius}: t = {t_stat:7.4f}, p = {p_val:.2e}")
            print(f"Significance: {sig}")
            print("-" * 80)
        
        print("\nSignificance levels: ns (p≥0.05), * (p<0.05), ** (p<0.01), *** (p<0.001)")
        print(f"(Note: Bonferroni correction applied for {n_comparisons} comparisons)")
        
        # Save statistics to file
        stats_file = os.path.join(output_dir, 
                                   f"{os.path.splitext(os.path.basename(excel_path))[0]}_BioRep_{bio_rep}_Statistics.txt")
        
        with open(stats_file, 'w') as f:
            f.write(f"Biological Replicate: {bio_rep}\n")
            f.write(f"Number of microglia: {n_microglia}\n")
            if filter_treatment:
                f.write(f"Treatment filter: {filter_treatment}\n")
            f.write("\nDESCRIPTIVE STATISTICS:\n")
            for s in stats_summary:
                f.write(f"Radius {s['Radius']:>2}: Mean={s['Mean']:10.4f}, SD={s['SD']:10.4f}, "
                       f"Min={s['Min']:10.4f}, Max={s['Max']:10.4f}\n")
            f.write(f"\nREPEATED MEASURES ANOVA:\n")
            f.write(f"F({df_between}, {df_within}) = {F_stat:.4f}, p = {p_value:.2e}\n")
            f.write(f"{interpretation}\n\n")
            f.write(f"PAIRWISE T-TESTS (Bonferroni corrected):\n")
            for pr in pairwise_results:
                f.write(f"{pr['Comparison']}: t={pr['t']:7.4f}, p={pr['p']:.2e}, {pr['sig']}\n")
        
        print(f"\nStatistics saved to {stats_file}")
        
        # Generate boxplot
        fig, ax = plt.subplots(figsize=(10, 6))
        
        positions = list(range(len(radius_list)))
        bp = ax.boxplot([data_dict[r] for r in radius_list], positions=positions,
                        patch_artist=True, widths=0.6)
        
        for patch in bp['boxes']:
            patch.set_facecolor('#4CAF50')
            patch.set_alpha(0.7)
        
        ax.set_xticks(positions)
        ax.set_xticklabels([f'r_{r}' for r in radius_list])
        ax.set_xlabel('Background Subtraction Radius (pixels)', fontsize=12)
        ax.set_ylabel('Area (pixels²)', fontsize=12)
        title_text = f'Biological Replicate {bio_rep}: Radius Comparison\n(n={n_microglia} microglia)'
        if filter_treatment:
            title_text += f' - {filter_treatment}'
        ax.set_title(title_text, fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)
        
        plot_file = os.path.join(output_dir,
                                f"{os.path.splitext(os.path.basename(excel_path))[0]}_BioRep_{bio_rep}_radius_comparison.png")
        plt.tight_layout()
        plt.savefig(plot_file, dpi=300)
        plt.close()
        
        print(f"Figure saved to {plot_file}")
        
        # Generate trajectory plot with significance markers
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # X-axis: radius values as numbers
        x_values = [float(r) for r in radius_list]
        
        # Plot each microglia as a separate line
        for i in range(n_microglia):
            y_values = [data_dict[r][i] for r in radius_list]
            ax.plot(x_values, y_values, marker='o', alpha=0.3, linewidth=1, markersize=4, color='gray')
        
        # Add mean line
        mean_values = [np.mean(data_dict[r]) for r in radius_list]
        ax.plot(x_values, mean_values, color='red', linewidth=3, marker='s', 
                markersize=8, label='Mean', zorder=100)
        
        # Add significance asterisks above mean line
        y_max = max(mean_values)
        y_range = max([max(data_dict[r]) for r in radius_list]) - min([min(data_dict[r]) for r in radius_list])
        annotation_height = y_max + 0.1 * y_range
        
        # Create significance dictionary
        sig_dict = {pr['Radius']: pr['sig'] for pr in pairwise_results}
        
        for i, radius in enumerate(radius_list):
            if radius in sig_dict and sig_dict[radius] != 'ns':
                ax.text(x_values[i], annotation_height, sig_dict[radius], 
                       ha='center', va='bottom', fontsize=14, fontweight='bold', color='black')
        
        ax.set_xlabel('Background Subtraction Radius (pixels)', fontsize=12)
        ax.set_ylabel('Area (pixels²)', fontsize=12)
        title_text = f'Biological Replicate {bio_rep}: Individual Microglia Trajectories\n(n={n_microglia} microglia)'
        if filter_treatment:
            title_text += f' - {filter_treatment}'
        ax.set_title(title_text, fontsize=14, fontweight='bold')
        ax.legend(fontsize=10, loc='upper left')
        ax.grid(True, alpha=0.3)
        
        # Add text annotation for reference
        ax.text(0.98, 0.02, f'Significance vs r_{reference_radius}\n* p<0.05, ** p<0.01, *** p<0.001', 
               transform=ax.transAxes, ha='right', va='bottom', 
               fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        trajectory_file = os.path.join(output_dir,
                                      f"{os.path.splitext(os.path.basename(excel_path))[0]}_BioRep_{bio_rep}_trajectories.png")
        plt.tight_layout()
        plt.savefig(trajectory_file, dpi=300)
        plt.close()
        
        print(f"Trajectory plot saved to {trajectory_file}")
    
    print(f"\n{'=' * 80}")
    print("ANALYSIS COMPLETE")
    print(f"All outputs saved to: {output_dir}")
    print("=" * 80)

# Example usage:
Analyze_Background_Subtraction_Stats(
    excel_path='/Users/allisonpickle/Desktop/Microglia Area_Background Sub.xlsx',
    sheet_name='Sheet1',
    filter_treatment='5ugLPS'  # Only process 5ugLPS images
)

In [ ]:
import os
import pandas as pd

def split_radii_to_files(
    input_excel_path,
    sheet_name=None,
    filename_col=0,
    overwrite=False,
    expected_radii=('0','10','18','25','35','50'),
    output_format='csv'  # NEW: 'csv' or 'excel'
):
    """
    Read an Excel sheet where:
      - Column 0: image filename (e.g., 240623_Tri_5ugLPS_10_20_2f)
      - Columns 1..N: values for radii (in order), e.g. r_0, r_10, r_18, ...
    For each radius column, export a NEW file with columns:
      Filename | Treatment | Biological_Rep | Well | Image | Results

    Filename format expected:
      date_experiment_treatment_wellid_dose_volume_imagetoken
      Example: 240623_Tri_5ugLPS_10_20_2f
      - parts[0] = date (240623) → Biological_Rep mapping
      - parts[1] = experiment (Tri)
      - parts[2] = treatment (5ugLPS) → Treatment
      - parts[3] = wellid (10) → Well_ID
      - parts[4] = dose (20)
      - parts[5] = imagetoken (2f) → ImageRaw

    Biological_Rep:
      - Determined from date token (parts[0]).
      - Unique dates are mapped to 1,2,3... in order of appearance.

    Well numbering:
      - Increments from 1 based on unique Well_ID values in order of appearance
      - First unique Well_ID → Well 1, Second unique Well_ID → Well 2, etc.
    """
    # Load sheet
    if sheet_name is None:
        df = pd.read_excel(input_excel_path)
        sheet_name = pd.ExcelFile(input_excel_path).sheet_names[0]
    else:
        df = pd.read_excel(input_excel_path, sheet_name=sheet_name)

    headers = list(df.columns)

    # Determine (radius, column_name) pairs
    radii_cols = []
    for i, col in enumerate(headers):
        if i == filename_col:
            continue
        name = str(col).strip().lower()
        if name.startswith('r_'):
            radii_cols.append((name[2:], col))
    if not radii_cols:
        subsequent_cols = [c for idx, c in enumerate(headers) if idx != filename_col]
        for i, r in enumerate(expected_radii):
            if i < len(subsequent_cols):
                radii_cols.append((r, subsequent_cols[i]))

    # Build biological replicate token mapping from date (parts[0]) -> 1..N
    first_tokens = []
    seen = set()
    for idx in range(len(df)):
        name = df.iloc[idx, filename_col]
        if pd.isna(name):
            continue
        base = os.path.splitext(os.path.basename(str(name).strip()))[0]
        parts = base.split('_')
        date_token = parts[0] if len(parts) > 0 else ""
        if date_token and date_token not in seen:
            seen.add(date_token)
            first_tokens.append(date_token)
    token_to_rep = {tok: i+1 for i, tok in enumerate(first_tokens)}

    def parse_rows(radius_value_col):
        rows = []
        for idx in range(len(df)):
            name = df.iloc[idx, filename_col]
            if pd.isna(name):
                continue
            name = str(name).strip()
            base = os.path.splitext(os.path.basename(name))[0]
            parts = base.split('_')

            # Expected: date_experiment_treatment_wellid_dose_volume_imagetoken
            date_token = parts[0] if len(parts) > 0 else ""
            experiment = parts[1] if len(parts) > 1 else ""
            treatment  = parts[2] if len(parts) > 2 else ""
            well_id    = parts[3] if len(parts) > 3 else ""
            dose       = parts[4] if len(parts) > 4 else ""
            volume     = parts[5] if len(parts) > 5 else ""
            image_tok  = parts[6] if len(parts) > 6 else (parts[5] if len(parts) > 5 else "")

            # numeric well number from parts[3]
            try:
                well_num = int(well_id) if well_id != "" else None
            except (ValueError, TypeError):
                well_num = None

            rows.append({
                "Filename": name,
                "Treatment": treatment,
                "Dose": dose,
                "Volume": volume,
                "Well_ID": well_id,
                "Well_Num": well_num,
                "ImageRaw": image_tok,
                "Biological_Rep": token_to_rep.get(date_token),
                "Results": df.iloc[idx][radius_value_col],
                "Original_Order": idx,
            })
        return pd.DataFrame(rows)

    def assign_well_and_image(out_df):
        # preserve original order
        out_df = out_df.sort_values("Original_Order").reset_index(drop=True)

        # Well number based on unique Well_ID in order of appearance
        unique_well_ids = []
        well_id_to_well = {}
        
        for well_id in out_df["Well_ID"]:
            if well_id not in well_id_to_well:
                unique_well_ids.append(well_id)
                well_id_to_well[well_id] = len(unique_well_ids)
        
        out_df["Well"] = out_df["Well_ID"].map(well_id_to_well)
        out_df["Image"] = out_df.groupby("Well_ID", sort=False).cumcount() + 1

        # Final selection, keep order
        out_df = out_df.sort_values("Original_Order")
        out_df = out_df[["Filename", "Treatment", "Biological_Rep", "Well", "Image", "Results"]]
        return out_df

    # Export separate files per radius
    in_dir = os.path.dirname(os.path.abspath(input_excel_path))
    in_stem = os.path.splitext(os.path.basename(input_excel_path))[0]

    written_files = []
    for r, col in radii_cols:
        tmp = parse_rows(radius_value_col=col)
        tmp = assign_well_and_image(tmp)

        # UPDATED: Build output path based on format
        if output_format.lower() == 'csv':
            out_path = os.path.join(in_dir, f"{in_stem}_r_{r}_RMixedFormatted.csv")
        else:
            out_path = os.path.join(in_dir, f"{in_stem}_r_{r}_RMixedFormatted.xlsx")

        if not overwrite and os.path.exists(out_path):
            base, ext = os.path.splitext(out_path)
            i = 1
            while True:
                candidate = f"{base} ({i}){ext}"
                if not os.path.exists(candidate):
                    out_path = candidate
                    break
                i += 1

        # UPDATED: Write as CSV or Excel
        if output_format.lower() == 'csv':
            tmp.to_csv(out_path, index=False)
        else:
            with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
                tmp.to_excel(writer, sheet_name="Data", index=False)

        print(f"✓ Written: {out_path}  rows={len(tmp)}")
        written_files.append(out_path)

    print(f"Done. Created {len(written_files)} file(s) as {output_format.upper()}:")
    for p in written_files:
        print(f"  - {os.path.basename(p)}")

    return written_files

# Example usage - CSV output:
files = split_radii_to_files(
    input_excel_path='/Users/allisonpickle/Desktop/Microglia Area_Background Sub.xlsx',
    sheet_name='Sheet1',
    output_format='csv'  # NEW: specify 'csv' or 'excel'
)

# Or for Excel output:
# files = split_radii_to_files(
#     input_excel_path='/Users/allisonpickle/Desktop/Microglia Area_Background Sub.xlsx',
#     sheet_name='Sheet1',
#     output_format='excel'
# )

In [1]:
import os
import pandas as pd

def combine_background_radii_to_single_file(
    input_excel_path,
    sheet_name=None,
    basename_col=0,
    overwrite=False,
    expected_radii=('0','10','18','25','35','50'),
    output_format='csv'
):
    """
    Read an Excel sheet with multiple background subtraction radius measurements.
    
    Input format:
      - Column 0: Filename (e.g., 240623_Tri_5ugLPS_2_1.tiff)
      - Columns 1-N: Background radius measurements (radius_0, radius_10, ..., radius_50)
    
    Output: Single CSV with columns:
      Filename | Treatment | Biological_Rep | Well | Image | Radius | Results

    Filename format:
      bioRep_wellID_treatment_imageNum.tiff
      Example: 240623_Tri_5ugLPS_2_1.tiff
      - parts[0] = biological replicate date (240623, 240624, 240625 → Bio Rep 1, 2, 3)
      - parts[2] = treatment (Ctrl, 5ugLPS, etc.)
      - Every 4 images = 1 Well (resets per treatment & radius)
    """
    # Load sheet
    if sheet_name is None:
        df = pd.read_excel(input_excel_path)
        sheet_name = pd.ExcelFile(input_excel_path).sheet_names[0]
    else:
        df = pd.read_excel(input_excel_path, sheet_name=sheet_name)

    headers = list(df.columns)

    # Identify radius columns
    radius_cols = []
    for i, col in enumerate(headers):
        if i == basename_col:
            continue
        name = str(col).strip().lower()
        if name.startswith('radius_') or name.startswith('radius') or name.startswith('bg_') or name.startswith('background_'):
            # Extract numeric radius value
            radius_val = name.replace('radius_', '').replace('radius', '').replace('bg_', '').replace('background_', '')
            radius_cols.append((radius_val, col))
    
    # Fallback: use expected radius order if columns aren't labeled
    if not radius_cols:
        subsequent_cols = [c for idx, c in enumerate(headers) if idx != basename_col]
        for i, r in enumerate(expected_radii):
            if i < len(subsequent_cols):
                radius_cols.append((r, subsequent_cols[i]))

    print(f"✓ Found {len(radius_cols)} radius columns: {[r[0] for r in radius_cols]}")

    # Build biological replicate mapping from unique date tokens
    first_tokens = []
    seen = set()
    for idx in range(len(df)):
        name = df.iloc[idx, basename_col]
        if pd.isna(name):
            continue
        base = str(name).strip()
        parts = base.split('_')
        bio_token = parts[0] if len(parts) > 0 else ""
        if bio_token and bio_token not in seen:
            seen.add(bio_token)
            first_tokens.append(bio_token)
    token_to_rep = {tok: i+1 for i, tok in enumerate(first_tokens)}

    print(f"✓ Biological replicate mapping: {token_to_rep}")

    # Parse all rows for all radius values
    all_rows = []
    
    for radius_value, radius_col in radius_cols:
        for idx in range(len(df)):
            name = df.iloc[idx, basename_col]
            if pd.isna(name):
                continue
            name = str(name).strip()
            parts = name.split('_')

            # Parse filename components
            bio_token = parts[0] if len(parts) > 0 else ""
            treatment = parts[2] if len(parts) > 2 else ""

            all_rows.append({
                "Filename": name,
                "Treatment": treatment,
                "Biological_Rep": token_to_rep.get(bio_token),
                "Radius": radius_value,
                "Results": df.iloc[idx][radius_col],
                "Original_Order": idx,
            })
    
    # Create combined dataframe
    combined_df = pd.DataFrame(all_rows)
    
    # Sort to maintain structure
    combined_df = combined_df.sort_values(["Radius", "Original_Order"]).reset_index(drop=True)
    
    # Assign Well and Image: every 4 images = 1 Well (resets per treatment & radius)
    combined_df["Well"] = 0
    combined_df["Image"] = 0
    
    for radius in combined_df["Radius"].unique():
        for treatment in combined_df["Treatment"].unique():
            mask = (combined_df["Radius"] == radius) & (combined_df["Treatment"] == treatment)
            treatment_rows = combined_df[mask]
            
            well_counter = 1
            for i, idx in enumerate(treatment_rows.index):
                combined_df.loc[idx, "Well"] = well_counter
                combined_df.loc[idx, "Image"] = ((i % 4) + 1)
                
                # Increment well every 4 images
                if (i + 1) % 4 == 0:
                    well_counter += 1
    
    # Final column selection
    combined_df = combined_df[["Filename", "Treatment", "Biological_Rep", "Well", "Image", "Radius", "Results"]]
    
    # Build output path
    in_dir = os.path.dirname(os.path.abspath(input_excel_path))
    in_stem = os.path.splitext(os.path.basename(input_excel_path))[0]
    base_name = f"{in_stem}_Combined_BackgroundRadius"
    
    if output_format.lower() == 'csv':
        out_path = os.path.join(in_dir, f"{base_name}_RMixedFormatted.csv")
    else:
        out_path = os.path.join(in_dir, f"{base_name}_RMixedFormatted.xlsx")

    # Handle existing files
    if not overwrite and os.path.exists(out_path):
        base, ext = os.path.splitext(out_path)
        i = 1
        while True:
            candidate = f"{base} ({i}){ext}"
            if not os.path.exists(candidate):
                out_path = candidate
                break
            i += 1

    # Write file
    if output_format.lower() == 'csv':
        combined_df.to_csv(out_path, index=False)
    else:
        with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
            combined_df.to_excel(writer, sheet_name="Data", index=False)

    # Summary
    print(f"\n{'='*60}")
    print(f"✓ COMBINED FILE CREATED: {out_path}")
    print(f"  Total rows: {len(combined_df)}")
    print(f"  Radius values: {sorted(combined_df['Radius'].unique())}")
    print(f"  Treatments: {sorted(combined_df['Treatment'].unique())}")
    print(f"  Biological replicates: {sorted(combined_df['Biological_Rep'].unique())}")
    print(f"  Wells per treatment×radius: {combined_df.groupby(['Treatment', 'Radius'])['Well'].max().to_dict()}")
    print(f"{'='*60}\n")
    
    return out_path


# ================================================================================
# USAGE
# ================================================================================
output_file = combine_background_radii_to_single_file(
    input_excel_path='/Users/allisonpickle/Desktop/Image Processing Pipeline Analysis/260110_Background Subtraction/Updated Background Sub Area.xlsx',
    sheet_name='Sheet1',
    output_format='csv',
    expected_radii=('0', '10', '18', '25', '35', '50')
)

print(f"Ready for R analysis! Feed this file to your mixed model script:")
print(f"  {output_file}")

✓ Found 6 radius columns: ['0', '10', '18', '25', '35', '50']
✓ Biological replicate mapping: {'240623': 1, '251104': 2, '260203': 3}

✓ COMBINED FILE CREATED: /Users/allisonpickle/Desktop/Image Processing Pipeline Analysis/260110_Background Subtraction/Updated Background Sub Area_Combined_BackgroundRadius_RMixedFormatted.csv
  Total rows: 1254
  Radius values: ['0', '10', '18', '25', '35', '50']
  Treatments: ['5ugLPS', 'Ctrl']
  Biological replicates: [1, 2, 3]
  Wells per treatment×radius: {('5ugLPS', '0'): 24, ('5ugLPS', '10'): 24, ('5ugLPS', '18'): 24, ('5ugLPS', '25'): 24, ('5ugLPS', '35'): 24, ('5ugLPS', '50'): 24, ('Ctrl', '0'): 29, ('Ctrl', '10'): 29, ('Ctrl', '18'): 29, ('Ctrl', '25'): 29, ('Ctrl', '35'): 29, ('Ctrl', '50'): 29}

Ready for R analysis! Feed this file to your mixed model script:
  /Users/allisonpickle/Desktop/Image Processing Pipeline Analysis/260110_Background Subtraction/Updated Background Sub Area_Combined_BackgroundRadius_RMixedFormatted.csv
